# QCD 2-to-2 Autoregressive Training

Single-GPU Kaggle run for the autoregressive graph-to-formula model.

In [1]:
!pip uninstall -y -q torchvision

from pathlib import Path

wheel_dirs = [p for p in Path('/kaggle/input').glob('**/wheels') if list(p.glob('*.whl'))]
if wheel_dirs:
    WHEEL_DIR = str(wheel_dirs[0])
    print('Installing from wheels:', WHEEL_DIR)
    !pip install -q --no-index --find-links "$WHEEL_DIR" pytorch-lightning torch-geometric
else:
    print('No wheel folder found; using pip index.')
    !pip install -q pytorch-lightning torch-geometric


Installing from wheels: /kaggle/input/notebooks/nextsmallestml/package/wheels
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires torchvision>=0.5, which is not installed.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have t

In [2]:
import os, shutil
from pathlib import Path

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

code_candidates = sorted(INPUT.glob('**/ampgnn/train.py'))
assert code_candidates, 'Could not find ampgnn/train.py in Kaggle input datasets'
src_repo = code_candidates[0].parent
dst_repo = WORK / 'ampgnn'
if dst_repo.exists():
    shutil.rmtree(dst_repo)
shutil.copytree(src_repo, dst_repo)
(dst_repo / '__init__.py').touch()
os.environ['PYTHONPATH'] = str(WORK)

data_candidates = [p for p in INPUT.glob('**/data') if (p / 'QCD').exists()]
assert data_candidates, 'Could not find data/QCD in Kaggle input datasets'
DATA_DIR = str(data_candidates[0])
RUN_DIR = WORK / 'runs' / 'qcd_2to2_ar'
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DATA_DIR'] = DATA_DIR
os.environ['RUN_DIR'] = str(RUN_DIR)

print('repo:', src_repo, '->', dst_repo)
print('DATA_DIR:', DATA_DIR)
print('RUN_DIR:', RUN_DIR)


repo: /kaggle/input/datasets/nextsmallestml/ampgnn-code-2-0/ampgnn -> /kaggle/working/ampgnn
DATA_DIR: /kaggle/input/datasets/nextsmallestml/qcd-dataset-1-0/data
RUN_DIR: /kaggle/working/runs/qcd_2to2_ar


In [3]:
import torch
print('cuda:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


cuda: True
gpu count: 1
0 NVIDIA RTX PRO 6000 Blackwell Server Edition


In [4]:
%%bash
set -euo pipefail
python -m ampgnn.train --model QCD --data_dir "$DATA_DIR" --out_dir "$RUN_DIR" \
  --orders tree --ranks 2_to_2 --decoder_mode ar --loss_mode ce \
  --accelerator gpu --devices 1 --strategy auto --precision bf16-mixed \
  --epochs 400 --batch_size 8 --lr 3e-4 --eta_min 1e-5 \
  --scheduler cosine_warmup --warmup_steps 100 --gradient_clip_val 1.0 \
  --weight_decay 0.03 --label_smoothing 0.01 --length_loss_weight 0.0 \
  --mass_rewrite_mode leg_sets --max_perms 6 \
  --enc_hid 192 --enc_layers 4 --enc_heads 8 --enc_dropout 0.1 \
  --d_model 384 --dec_layers 4 --dec_nhead 8 --dec_dropout 0.1 \
  --dec_max_len 1024 --max_seq_len_cap 1024 --num_workers 2 \
  --print_every_n_epochs 1 \
  --fail_log_dir "$RUN_DIR/failures" --fail_log_max 200 \
  --no-auto_resume 2>&1 | tee "$RUN_DIR/train.log"


Seed set to 42
/kaggle/working/ampgnn/model.py:75: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(self.pool_gate)
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/call

In [5]:
!find "$RUN_DIR" -maxdepth 3 -type f | sort


/kaggle/working/runs/qcd_2to2_ar/failures/test_failures.jsonl
/kaggle/working/runs/qcd_2to2_ar/last.ckpt
/kaggle/working/runs/qcd_2to2_ar/QCD-91-0.0000.ckpt
/kaggle/working/runs/qcd_2to2_ar/run_state.json
/kaggle/working/runs/qcd_2to2_ar/train.log
